# 02 — Stage-2 DR Severity Grading: 4-fold CV, 512 px → 768 px

Thin driver around `src/training/stage2.py`; hyper-parameters live in `configs/tier3_stage2_grading.yaml`.

- **Pretrained backbone** + GeM pooling + CORAL tri-head (ordinal / referable / continuous).
- **True out-of-fold predictions:** one model per validation fold 1–4 (fold 0 stays the locked test split). These OOF predictions feed calibration in notebook 03, and the 4 fold models are ensembled at inference.
- **One** imbalance correction (class-balanced sampler by default), gradient accumulation (effective batch 32), warmup + cosine LR, EMA weights, fp16 loss scaling.
- **Session budget:** each fold is resumable and is skipped once finished (state mirrored to Drive). If a Colab session cannot fit all folds, set `FOLDS` to e.g. `[1, 2]` now and `[3, 4]` in the next session.

In [ ]:
# 1. Mount Google Drive, clone/update the repo and unpack the preprocessed data to local disk
import os, sys, subprocess, tarfile, zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/Retinopathy')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/dprincepiriyan/Retinopathy.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.8'], check=True)
    LOCAL_DATA_ROOT = Path('/content/data/processed')
    NEEDS_DATA = True
    if NEEDS_DATA and not (LOCAL_DATA_ROOT / 'stage2_grading_splits.csv').exists():
        drive_root = Path('/content/drive/MyDrive')
        candidates = list(drive_root.rglob('retinopathy_processed.tar.gz')) + list(drive_root.rglob('retinopathy_processed.zip'))
        if not candidates:
            raise FileNotFoundError('Upload retinopathy_processed.tar.gz to your Google Drive first.')
        archive_path = candidates[0]
        print(f'Extracting {archive_path} -> /content/data/ ...')
        Path('/content/data').mkdir(parents=True, exist_ok=True)
        if archive_path.suffix == '.zip':
            with zipfile.ZipFile(archive_path) as zf:
                zf.extractall('/content/data')
        else:
            with tarfile.open(archive_path, 'r:gz') as tf:
                tf.extractall('/content/data', filter='data')
    DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/Retinopathy_Checkpoints')
else:
    REPO_DIR = Path.cwd().resolve()
    if REPO_DIR.name == 'colab':
        REPO_DIR = REPO_DIR.parent
    LOCAL_DATA_ROOT = REPO_DIR / 'data' / 'processed'
    DRIVE_CKPT_DIR = None

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
LOCAL_CKPT_DIR = REPO_DIR / 'checkpoints'
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_CKPT_DIR is not None:
    DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

from src.utils.tpu_accelerator import ColabTPUAccelerator
accel = ColabTPUAccelerator(prefer_tpu=False)  # use a GPU runtime (T4 / L4 / A100)
print(f'Accelerator: {accel.info.device_name} | AMP dtype: {accel.info.amp_dtype} | GradScaler: {accel.grad_scaler is not None}')
if accel.info.backend != 'cuda':
    print('WARNING: no GPU detected - Runtime > Change runtime type > GPU. CPU training will be extremely slow.')

In [ ]:
# 2. Load the config
from src.utils.config import load_yaml_config
from src.training.cli import apply_overrides

cfg = load_yaml_config('configs/tier3_stage2_grading.yaml')['stage2_grading']
cfg = apply_overrides(cfg, [
    # 'training.amp=false',          # fp32 fallback if you ever get NaN/inf errors (slower)
    # 'model.backbone_name=convnext_tiny',
    # 'imbalance.strategy=loss_weights',
])
FOLDS = cfg['cv']['folds']   # e.g. [1, 2] in one session and [3, 4] in the next
cfg

In [ ]:
# 3. Train every requested fold (finished folds are skipped) and aggregate when all folds are done
from src.training.stage2 import run_stage2_cv

summary = run_stage2_cv(cfg, LOCAL_DATA_ROOT, LOCAL_CKPT_DIR, mirror_dir=DRIVE_CKPT_DIR, folds=FOLDS, accel=accel)

In [ ]:
# 4. Results (only available once ALL folds in cfg['cv']['folds'] are finished)
import json
if summary is None:
    print('Some folds are still missing - run this notebook again in a new session to continue.')
else:
    print('OOF (train_val) metrics:'); print(json.dumps(summary['oof_metrics'], indent=2))
    print('\nLocked TEST split, 4-fold ensemble:'); print(json.dumps(summary['test_ensemble_metrics'], indent=2))